# Gold - dim_date

This notebook seeds `amperedrive_giga.gold.dim_date`, a calendar spine from 2025-01-01 to 2030-12-31 with one row per day. Load pattern: Static / One-time Seed. It is idempotent: an insert-only MERGE on `date_key` adds only missing dates. Source: STTM - Gold Dimensions.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS amperedrive_giga.gold.dim_date (
  date_key    INT     NOT NULL COMMENT 'PK - yyyyMMdd integer',
  full_date   DATE    NOT NULL COMMENT 'NK - calendar date',
  year        INT     NOT NULL,
  quarter     INT     NOT NULL,
  month       INT     NOT NULL,
  month_name  STRING  NOT NULL,
  day_of_week STRING  NOT NULL,
  is_weekend  BOOLEAN NOT NULL,
  CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
)
COMMENT 'Gold calendar dimension (2025-01-01 to 2030-12-31). Static seed.'

In [0]:
%sql
MERGE INTO amperedrive_giga.gold.dim_date AS t
USING (
  SELECT
    CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT)                 AS date_key,
    full_date,
    YEAR(full_date)                                                 AS year,
    QUARTER(full_date)                                              AS quarter,
    MONTH(full_date)                                                AS month,
    DATE_FORMAT(full_date, 'MMMM')                                  AS month_name,
    DATE_FORMAT(full_date, 'EEEE')                                  AS day_of_week,
    CASE WHEN DAYOFWEEK(full_date) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend
  FROM (
    SELECT EXPLODE(SEQUENCE(DATE'2025-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS full_date
  )
) AS s
ON t.date_key = s.date_key
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
-- Expect 2191 rows (2025-01-01..2030-12-31), unique keys, and correct min/max dates
SELECT
  COUNT(*)                                   AS row_count,
  COUNT(DISTINCT date_key)                   AS distinct_keys,
  MIN(full_date)                             AS min_date,
  MAX(full_date)                             AS max_date,
  SUM(CASE WHEN is_weekend THEN 1 ELSE 0 END) AS weekend_days,
  assert_true(COUNT(*) = COUNT(DISTINCT date_key), 'dim_date has duplicate date_key') AS dq_unique_key
FROM amperedrive_giga.gold.dim_date